In [ ]:
# ============================================================
# ROUNDOFF NOISE IN DFT AND FFT COMPUTATION
# ============================================================
#
# This notebook studies the accumulation of fixed-point
# roundoff noise in direct DFT and FFT computation.
#
# The analysis follows the PQN model in which every quantizer
# is replaced by an additive, zero-mean, uncorrelated
# quantization-noise source.
#
#
# ============================================================
# QUANTIZATION-NOISE MODEL
# ============================================================
#
# With K fractional bits,
#
#                   Delta = 2^(-K)
#
# and the variance of one rounding-noise source is
#
#                   sigma_e^2 = Delta^2 / 12
#
#                             = 2^(-2K) / 12.
#
#
# ============================================================
# DIRECT DFT
# ============================================================
#
# For an N-point complex DFT, 4N real multiplications are
# associated with 4N quantization-noise sources.
#
# Assuming that all noise sources are mutually uncorrelated,
#
#       sigma_DFT^2
#
#       = 4N * Delta^2 / 12
#
#       = N Delta^2 / 3
#
#       = (N/3) 2^(-2K).
#
# Thus, for fixed K,
#
#                   sigma_DFT^2 proportional to N.
#
#
# ============================================================
# FFT WITHOUT STAGE SCALING
# ============================================================
#
# For a radix-2 FFT with
#
#                   N = 2^nu,
#
# the accumulated number of relevant multiplication-error
# contributions to one output sample leads to
#
#       sigma_FFT^2
#
#       = 4(N-1) Delta^2 / 12
#
#       = (N-1) Delta^2 / 3.
#
# For sufficiently large N,
#
#       sigma_FFT^2 approximately N Delta^2 / 3.
#
# Therefore, although the FFT requires significantly fewer
# multiplications than direct DFT computation, its accumulated
# roundoff-noise variance is of approximately the same order.
#
#
# ============================================================
# FFT WITH 1/2 SCALING AT EACH STAGE
# ============================================================
#
# To reduce overflow risk, each FFT stage can be scaled by 1/2.
#
# Since a factor 1/2 in amplitude reduces variance by 1/4,
# errors introduced early in the FFT are progressively
# attenuated as they propagate through the remaining stages.
#
# The resulting total output-noise variance is
#
#       sigma_scaled^2
#
#       = (2 Delta^2 / 3) [1 - (1/2)^nu]
#
#       = (2 Delta^2 / 3) [1 - 1/N].
#
# For large N,
#
#       sigma_scaled^2 approximately 2 Delta^2 / 3
#
#                         = (2/3) 2^(-2K).
#
# Thus, unlike the unscaled case, the scaled FFT noise variance
# is no longer proportional to N.
#
#
# ============================================================
# REQUIRED WORD LENGTH
# ============================================================
#
# For the direct DFT,
#
#       sigma_DFT^2 = (N/3) 2^(-2K).
#
# If N is multiplied by four,
#
#                   N -> 4N,
#
# maintaining the same noise variance requires
#
#                   K -> K + 1.
#
# Therefore, every fourfold increase in transform length
# requires approximately one additional fractional bit.
#
#
# ============================================================
# SNR FOR THE SCALED FFT
# ============================================================
#
# Using the relations given in the theoretical development:
#
#                   sigma_x^2 = 1 / (3N)
#
# and approximately
#
#                   sigma_e^2 = (2/3) 2^(-2K),
#
# the SNR becomes
#
#       SNR = sigma_x^2 / sigma_e^2
#
#           = [1/(3N)] / [(2/3) 2^(-2K)]
#
#           = 2^(2K) / (2N).
#
# Since
#
#                   N = 2^nu,
#
# it follows that
#
#                   SNR = 2^(2K - nu - 1).
#
#
# ============================================================
# HOW TO USE THIS NOTEBOOK
# ============================================================
#
# Select one of the following displays.
#
#
# 1. DIRECT DFT NOISE VARIANCE
#
#    Active controls:
#
#       K
#       nu
#
#    Observe the direct-DFT roundoff-noise variance for the
#    selected transform length
#
#                   N = 2^nu.
#
#
# 2. FFT WITHOUT SCALING
#
#    Active controls:
#
#       K
#       nu
#
#    Observe the accumulated FFT roundoff-noise variance without
#    stage-by-stage scaling.
#
#
# 3. FFT WITH STAGE SCALING
#
#    Active controls:
#
#       K
#       nu
#
#    Observe the variance obtained when every FFT stage is
#    multiplied by 1/2.
#
#
# 4. DIRECT DFT VS FFT
#
#    Active control:
#
#       K
#
#    The three theoretical curves are compared as N increases.
#
#
# 5. REQUIRED BITS VS N
#
#    Active control:
#
#       target variance
#
#    The fractional word length K required to keep the direct-DFT
#    roundoff-noise variance below a specified target is shown.
#
#
# 6. SNR WITH STAGE SCALING
#
#    Active controls:
#
#       K
#       nu
#
#    Observe the theoretical SNR
#
#                   SNR = 2^(2K - nu - 1).
#
#
# ============================================================
# WHAT WE EXPECT TO OBSERVE
# ============================================================
#
# DIRECT DFT
#
# For fixed K, increasing N should increase the noise variance
# almost linearly because
#
#                   sigma_DFT^2 proportional to N.
#
#
# FFT WITHOUT SCALING
#
# The variance should behave almost identically to the direct DFT
# for sufficiently large N:
#
#                   sigma_FFT^2 approximately sigma_DFT^2.
#
# This is an important result: fewer arithmetic operations do not
# automatically imply proportionally smaller accumulated
# roundoff noise.
#
#
# FFT WITH STAGE SCALING
#
# The variance should initially increase with N but should rapidly
# approach the asymptotic value
#
#                   (2/3) 2^(-2K).
#
# Thus, the scaled FFT curve should flatten instead of continuing
# to grow proportionally to N.
#
#
# DIRECT DFT VS FFT
#
# The direct DFT and unscaled FFT curves should remain close to
# each other and grow with N.
#
# The scaled FFT curve should remain dramatically lower for
# sufficiently large N.
#
#
# REQUIRED BITS VS N
#
# Increasing N by a factor of four should require approximately
# one additional fractional bit to preserve a fixed roundoff-noise
# variance.
#
#
# SNR
#
# Increasing K improves the SNR very rapidly.
#
# Increasing nu, and therefore N, decreases the SNR.
#
# Since
#
#                   SNR = 2^(2K - nu - 1),
#
# one additional fractional bit increases the SNR by a factor
# of four.
#
#
# ============================================================
# IMPORTANT INTERPRETATION
# ============================================================
#
# This notebook does not simulate a complete FFT implementation.
#
# It visualizes the analytical roundoff-noise relations derived
# from the PQN model.
#
# The central purpose is to demonstrate how:
#
#       transform length,
#       word length,
#       FFT structure,
#       and stage scaling
#
# affect accumulated roundoff-noise variance.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatLogSlider, RadioButtons, VBox, HBox, HTML, Layout, interactive
from IPython.display import display


# ------------------------------------------------------------
# Basic theoretical functions
# ------------------------------------------------------------

def quantization_step(K):

    return 2.0**(-K)


def one_source_variance(K):

    Delta = quantization_step(K)

    return Delta**2 / 12.0


def direct_dft_variance(N, K):

    Delta = quantization_step(K)

    return (N / 3.0) * Delta**2


def fft_unscaled_variance(N, K):

    Delta = quantization_step(K)

    return ((N - 1.0) / 3.0) * Delta**2


def fft_scaled_variance(N, K):

    Delta = quantization_step(K)

    return (2.0 / 3.0) * Delta**2 * (1.0 - 1.0 / N)


def fft_scaled_asymptotic_variance(K):

    Delta = quantization_step(K)

    return (2.0 / 3.0) * Delta**2


def required_bits_direct_dft(N, target_variance):

    K_real = 0.5 * np.log2(N / (3.0 * target_variance))

    return K_real


def scaled_fft_snr_linear(N, K):

    return (2.0**(2 * K)) / (2.0 * N)


def scaled_fft_snr_db(N, K):

    snr_linear = scaled_fft_snr_linear(N, K)

    return 10.0 * np.log10(snr_linear)


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.df-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.df-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.df-howto {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7cfe0;
    border-left: 6px solid #667da8;
    background: #f8f9fc;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.df-howto-title {
    font-size: 13px;
    font-weight: bold;
    color: #40587d;
    margin-bottom: 5px;
}

.df-observe {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7d8c9;
    border-left: 6px solid #3c8a4e;
    background: #f7fbf7;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.df-observe-title {
    font-size: 13px;
    font-weight: bold;
    color: #245c31;
    margin-bottom: 5px;
}

.df-model {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #d5c58a;
    border-left: 6px solid #b8860b;
    background: #fffaf0;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.df-model-title {
    font-size: 13px;
    font-weight: bold;
    color: #8a6500;
    margin-bottom: 5px;
}

.df-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.df-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.df-info {
    font-size: 13px;
    line-height: 1.52;
}

.df-label {
    display: inline-block;
    min-width: 245px;
    font-weight: bold;
}

.df-value {
    font-size: 14px;
    font-weight: bold;
}

.df-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 6px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="df-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Roundoff Noise in DFT and FFT Computation
    </div>

</div>
""")


# ------------------------------------------------------------
# Visible introductory documentation
# ------------------------------------------------------------

description_html = HTML("""
<div class="df-root">

    <div class="df-description">

        <b>What this notebook demonstrates</b><br><br>

        Fixed-point DFT and FFT implementations introduce rounding errors
        whenever multiplication results are quantized. Under the PQN model,
        each quantizer is replaced by an independent additive noise source
        with

        <div style="text-align:center; margin:6px 0;">
            <b>
            Δ = 2<sup>−K</sup>,
            &nbsp;&nbsp;
            σ²<sub>q</sub> = Δ²/12.
            </b>
        </div>

        The notebook compares three theoretical situations:

        <div style="text-align:center; margin:7px 0;">
            <b>
            Direct DFT:
            σ² = (N/3)2<sup>−2K</sup>
            </b>
        </div>

        <div style="text-align:center; margin:7px 0;">
            <b>
            FFT without scaling:
            σ² = [(N−1)/3]2<sup>−2K</sup>
            </b>
        </div>

        <div style="text-align:center; margin:7px 0;">
            <b>
            FFT with 1/2 stage scaling:
            σ² = (2/3)2<sup>−2K</sup>(1−1/N).
            </b>
        </div>

    </div>


    <div class="df-howto">

        <div class="df-howto-title">
            How to use this notebook
        </div>

        <b>Direct DFT noise variance:</b>
        change K and ν, where N = 2<sup>ν</sup>, and inspect the accumulated
        direct-DFT roundoff-noise variance.<br><br>

        <b>FFT without scaling:</b>
        change K and ν and observe the corresponding unscaled FFT noise
        variance.<br><br>

        <b>FFT with stage scaling:</b>
        change K and ν and observe how 1/2 scaling at each FFT stage prevents
        the variance from continuing to grow proportionally with N.<br><br>

        <b>Direct DFT vs FFT:</b>
        choose K and compare all three theoretical curves over a wide range
        of transform lengths.<br><br>

        <b>Required bits vs N:</b>
        choose the maximum acceptable noise variance and observe the
        fractional word length required as N increases.<br><br>

        <b>SNR with stage scaling:</b>
        change K and ν and observe both the linear and dB values of the
        theoretical SNR.<br><br>

        Controls that do not affect the selected display are automatically
        disabled.

    </div>


    <div class="df-observe">

        <div class="df-observe-title">
            What we expect to observe
        </div>

        <b>Direct DFT:</b>
        for fixed K, the variance increases proportionally with N.<br><br>

        <b>FFT without scaling:</b>
        despite requiring fewer multiplications, its accumulated roundoff
        variance remains approximately equal to the direct-DFT result for
        sufficiently large N.<br><br>

        <b>FFT with stage scaling:</b>
        the variance should approach a limiting value instead of continuing
        to grow with N. This is the main visual effect of distributed
        1/2 scaling.<br><br>

        <b>Required bits:</b>
        every fourfold increase in N should require approximately one
        additional fractional bit to preserve the same direct-DFT noise
        variance.<br><br>

        <b>SNR:</b>
        increasing K strongly improves SNR, whereas increasing N reduces it.
        One additional fractional bit increases the theoretical SNR by a
        factor of four.

    </div>


    <div class="df-model">

        <div class="df-model-title">
            Main theoretical conclusion
        </div>

        Without stage scaling, both direct DFT and FFT roundoff-noise
        variances are approximately proportional to N:

        <div style="text-align:center; margin:6px 0;">
            <b>
            σ² ≈ (N/3)2<sup>−2K</sup>.
            </b>
        </div>

        With 1/2 scaling at every FFT stage,

        <div style="text-align:center; margin:6px 0;">
            <b>
            σ² → (2/3)2<sup>−2K</sup>
            </b>
        </div>

        as N becomes large. Thus, the dependence on transform length is
        essentially removed from the asymptotic noise variance.

    </div>

</div>
""")


# ------------------------------------------------------------
# Dynamic numerical summary
# ------------------------------------------------------------

summary_html = HTML()

summary_html.layout = Layout(width='610px', min_width='610px', overflow='visible')


# ------------------------------------------------------------
# Main interactive function
# ------------------------------------------------------------

def plot_dft_fft_roundoff(display_mode='Direct DFT noise variance', K=12, nu=8, target_variance=1e-6):

    N = 2**nu

    Delta = quantization_step(K)

    sigma_source = one_source_variance(K)

    sigma_dft = direct_dft_variance(N, K)

    sigma_fft = fft_unscaled_variance(N, K)

    sigma_scaled = fft_scaled_variance(N, K)

    sigma_scaled_limit = fft_scaled_asymptotic_variance(K)

    snr_linear = scaled_fft_snr_linear(N, K)

    snr_db = scaled_fft_snr_db(N, K)

    K_required_real = required_bits_direct_dft(N, target_variance)

    K_required_integer = int(np.ceil(K_required_real))


    # --------------------------------------------------------
    # Dynamic summary
    # --------------------------------------------------------

    if display_mode == 'Required bits vs N':

        mode_information = f"""
        <span class="df-label">Target variance</span>
        {target_variance:.3e}
        <br>

        <span class="df-label">Selected transform length</span>
        N = 2<sup>{nu}</sup> = {N}
        <br>

        <span class="df-label">Required K, continuous</span>
        {K_required_real:.4f}
        <br>

        <span class="df-label">Required K, integer</span>
        <span class="df-value">{K_required_integer} bits</span>
        """

    elif display_mode == 'SNR with stage scaling':

        mode_information = f"""
        <span class="df-label">Transform length</span>
        N = 2<sup>{nu}</sup> = {N}
        <br>

        <span class="df-label">Fractional bits</span>
        K = {K}
        <br>

        <span class="df-label">Linear SNR</span>
        <span class="df-value">{snr_linear:.8e}</span>
        <br>

        <span class="df-label">SNR in dB</span>
        <span class="df-value">{snr_db:.4f} dB</span>
        """

    else:

        mode_information = f"""
        <span class="df-label">Transform length</span>
        N = 2<sup>{nu}</sup> = <span class="df-value">{N}</span>
        <br>

        <span class="df-label">Fractional bits</span>
        K = <span class="df-value">{K}</span>
        <br>

        <span class="df-label">Quantization step</span>
        Δ = {Delta:.8e}
        <br>

        <span class="df-label">One-source variance</span>
        Δ²/12 = {sigma_source:.8e}
        <br><br>

        <span class="df-label">Direct DFT variance</span>
        {sigma_dft:.8e}
        <br>

        <span class="df-label">FFT unscaled variance</span>
        {sigma_fft:.8e}
        <br>

        <span class="df-label">FFT scaled variance</span>
        {sigma_scaled:.8e}
        <br>

        <span class="df-label">Scaled asymptotic limit</span>
        {sigma_scaled_limit:.8e}
        """


    summary_html.value = f"""
    <div class="df-box">

        <div class="df-title">
            Current Roundoff-Noise Data
        </div>

        <div class="df-info">

            <span class="df-label">Selected display</span>
            {display_mode}
            <br><br>

            {mode_information}

        </div>

        <div class="df-note">
            All displayed variance relations follow the analytical PQN model.
            Controls that do not affect the selected representation are
            automatically disabled.
        </div>

    </div>
    """


    # --------------------------------------------------------
    # One large figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(11.8, 5.0))


    # ========================================================
    # DIRECT DFT NOISE VARIANCE
    # ========================================================

    if display_mode == 'Direct DFT noise variance':

        nu_values = np.arange(2, 15)

        N_values = 2**nu_values

        variances = np.array([direct_dft_variance(n_value, K) for n_value in N_values])


        ax.plot(N_values, variances, 'o-', linewidth=1.8, markersize=5, label='Direct DFT')

        ax.plot(N, sigma_dft, 's', markersize=9, label='Current N')


        ax.set_xscale('log', base=2)

        ax.set_yscale('log')


        ax.set_xticks(N_values)

        ax.set_xticklabels([str(value) for value in N_values], rotation=45)


        ax.set_xlabel('Transform length N')

        ax.set_ylabel('Roundoff-noise variance')

        ax.set_title('Direct DFT Roundoff-Noise Variance', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False, fontsize=9)


    # ========================================================
    # FFT WITHOUT SCALING
    # ========================================================

    elif display_mode == 'FFT without scaling':

        nu_values = np.arange(2, 15)

        N_values = 2**nu_values

        variances = np.array([fft_unscaled_variance(n_value, K) for n_value in N_values])


        ax.plot(N_values, variances, 'o-', linewidth=1.8, markersize=5, label='FFT without stage scaling')

        ax.plot(N, sigma_fft, 's', markersize=9, label='Current N')


        ax.set_xscale('log', base=2)

        ax.set_yscale('log')


        ax.set_xticks(N_values)

        ax.set_xticklabels([str(value) for value in N_values], rotation=45)


        ax.set_xlabel('Transform length N')

        ax.set_ylabel('Roundoff-noise variance')

        ax.set_title('FFT Roundoff Noise without Stage Scaling', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False, fontsize=9)


    # ========================================================
    # FFT WITH STAGE SCALING
    # ========================================================

    elif display_mode == 'FFT with stage scaling':

        nu_values = np.arange(2, 15)

        N_values = 2**nu_values

        variances = np.array([fft_scaled_variance(n_value, K) for n_value in N_values])

        asymptotic_values = np.full_like(variances, sigma_scaled_limit, dtype=float)


        ax.plot(N_values, variances, 'o-', linewidth=1.8, markersize=5, label='FFT with 1/2 scaling per stage')

        ax.plot(N_values, asymptotic_values, '--', linewidth=1.5, label='Asymptotic limit')

        ax.plot(N, sigma_scaled, 's', markersize=9, label='Current N')


        ax.set_xscale('log', base=2)


        ax.set_xticks(N_values)

        ax.set_xticklabels([str(value) for value in N_values], rotation=45)


        ax.set_xlabel('Transform length N')

        ax.set_ylabel('Roundoff-noise variance')

        ax.set_title('FFT Roundoff Noise with 1/2 Stage Scaling', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=9)


    # ========================================================
    # DIRECT DFT VS FFT
    # ========================================================

    elif display_mode == 'Direct DFT vs FFT':

        nu_values = np.arange(2, 15)

        N_values = 2**nu_values


        variance_dft = np.array([direct_dft_variance(n_value, K) for n_value in N_values])

        variance_fft = np.array([fft_unscaled_variance(n_value, K) for n_value in N_values])

        variance_scaled = np.array([fft_scaled_variance(n_value, K) for n_value in N_values])


        ax.plot(N_values, variance_dft, 'o-', linewidth=1.8, markersize=4, label='Direct DFT')

        ax.plot(N_values, variance_fft, 's--', linewidth=1.8, markersize=4, label='FFT without scaling')

        ax.plot(N_values, variance_scaled, '^-.', linewidth=1.8, markersize=4, label='FFT with 1/2 stage scaling')


        ax.set_xscale('log', base=2)

        ax.set_yscale('log')


        ax.set_xticks(N_values)

        ax.set_xticklabels([str(value) for value in N_values], rotation=45)


        ax.set_xlabel('Transform length N')

        ax.set_ylabel('Roundoff-noise variance')

        ax.set_title(f'Direct DFT and FFT Roundoff Noise — K = {K}', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=9)


    # ========================================================
    # REQUIRED BITS VS N
    # ========================================================

    elif display_mode == 'Required bits vs N':

        nu_values = np.arange(2, 17)

        N_values = 2**nu_values


        K_real_values = np.array([required_bits_direct_dft(n_value, target_variance) for n_value in N_values])

        K_integer_values = np.ceil(K_real_values)


        ax.plot(N_values, K_real_values, 'o-', linewidth=1.8, markersize=4, label='Continuous required K')

        ax.step(N_values, K_integer_values, where='mid', linewidth=1.8, label='Integer required K')


        ax.plot(N, K_required_real, 's', markersize=9, label='Current N')


        ax.set_xscale('log', base=2)


        ax.set_xticks(N_values)

        ax.set_xticklabels([str(value) for value in N_values], rotation=45)


        ax.set_xlabel('Transform length N')

        ax.set_ylabel('Required fractional bits K')

        ax.set_title(f'Required Word Length for Target Variance = {target_variance:.1e}', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=9)


    # ========================================================
    # SNR WITH STAGE SCALING
    # ========================================================

    else:

        K_values = np.arange(2, 21)

        snr_db_values = np.array([scaled_fft_snr_db(N, k_value) for k_value in K_values])


        ax.plot(K_values, snr_db_values, 'o-', linewidth=1.8, markersize=5, label=f'N = {N}')

        ax.plot(K, snr_db, 's', markersize=9, label='Current K')


        ax.set_xticks(K_values)


        ax.set_xlabel('Fractional bits K')

        ax.set_ylabel('Theoretical SNR (dB)')

        ax.set_title(f'SNR with FFT Stage Scaling — N = {N}', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False, fontsize=9)


    # --------------------------------------------------------
    # Final spacing
    # --------------------------------------------------------

    plt.subplots_adjust(left=0.09, right=0.98, top=0.90, bottom=0.27)

    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(width='295px')

slider_style = {'description_width': '120px'}


display_selector = RadioButtons(
    options=[
        'Direct DFT noise variance',
        'FFT without scaling',
        'FFT with stage scaling',
        'Direct DFT vs FFT',
        'Required bits vs N',
        'SNR with stage scaling'
    ],
    value='Direct DFT noise variance',
    description='Display:',
    style={'description_width': '65px'},
    layout=Layout(width='310px')
)


K_slider = IntSlider(
    value=12,
    min=2,
    max=20,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


nu_slider = IntSlider(
    value=8,
    min=2,
    max=16,
    step=1,
    description='Exponent nu:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


target_variance_slider = FloatLogSlider(
    value=1e-6,
    base=10,
    min=-12,
    max=-2,
    step=0.25,
    description='Target variance:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2e'
)


# ------------------------------------------------------------
# Enable only relevant controls
# ------------------------------------------------------------

def update_control_states(change=None):

    mode = display_selector.value


    if mode == 'Direct DFT noise variance':

        K_slider.disabled = False
        nu_slider.disabled = False
        target_variance_slider.disabled = True


    elif mode == 'FFT without scaling':

        K_slider.disabled = False
        nu_slider.disabled = False
        target_variance_slider.disabled = True


    elif mode == 'FFT with stage scaling':

        K_slider.disabled = False
        nu_slider.disabled = False
        target_variance_slider.disabled = True


    elif mode == 'Direct DFT vs FFT':

        K_slider.disabled = False
        nu_slider.disabled = True
        target_variance_slider.disabled = True


    elif mode == 'Required bits vs N':

        K_slider.disabled = True
        nu_slider.disabled = False
        target_variance_slider.disabled = False


    else:

        K_slider.disabled = False
        nu_slider.disabled = False
        target_variance_slider.disabled = True


display_selector.observe(update_control_states, names='value')


# ------------------------------------------------------------
# Initial state
# ------------------------------------------------------------

update_control_states()


# ------------------------------------------------------------
# Interactive object
# ------------------------------------------------------------

widget_plot = interactive(
    plot_dft_fft_roundoff,
    display_mode=display_selector,
    K=K_slider,
    nu=nu_slider,
    target_variance=target_variance_slider
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls_box = VBox(
    [
        HTML("<div class='df-title'>Controls</div>"),
        display_selector,
        K_slider,
        nu_slider,
        target_variance_slider
    ],
    layout=Layout(
        width='335px',
        min_width='335px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Summary + controls
# ------------------------------------------------------------

top_row = HBox(
    [
        summary_html,
        controls_box
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Plot output
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

plot_output.layout = Layout(width='auto', overflow='visible')


# ------------------------------------------------------------
# Final layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)